In [0]:
import pandas as pd
from datetime import datetime, timezone
from sklearn.ensemble import HistGradientBoostingRegressor

FEATURES = ["hour_of_day", "pm25_now", "pm25_lag1", "pm25_lag3", "pm25_lag6", "pm25_lag24",
            "temperature_c", "humidity_pct", "wind_speed_kmh", "wind_direction_deg",
            "pressure_hpa", "precipitation_mm", "boundary_layer_m"]
TRAIN_BEFORE = "2026-10-31 18:00:00"   # frozen model: trained on data before Nov 1 (6-hour gap)
LOG_FROM = "2026-11-24 15:00:00"      # first hour whose 6-hour-ahead value was still unknown
LOG = "workspace.delhi_air.forecast_log"

train = spark.sql(f"""
    SELECT * FROM workspace.delhi_air.ml_features_6h
    WHERE hour_ist < TIMESTAMP '{TRAIN_BEFORE}'
""").toPandas()
model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=0)
model.fit(train[FEATURES], train["pm25_target_6h"])
print("trained on rows:", len(train))

live = spark.sql(f"""
WITH b AS (
    SELECT sensor_id, location_id, hour_ist, pm25_avg,
           temperature_c, humidity_pct, wind_speed_kmh, wind_direction_deg,
           pressure_hpa, precipitation_mm, boundary_layer_m
    FROM workspace.delhi_air.gold_station_hourly
)
SELECT b.sensor_id, b.location_id, b.hour_ist,
       hour(b.hour_ist) AS hour_of_day,
       b.pm25_avg AS pm25_now,
       l1.pm25_avg AS pm25_lag1, l3.pm25_avg AS pm25_lag3,
       l6.pm25_avg AS pm25_lag6, l24.pm25_avg AS pm25_lag24,
       b.temperature_c, b.humidity_pct, b.wind_speed_kmh, b.wind_direction_deg,
       b.pressure_hpa, b.precipitation_mm, b.boundary_layer_m
FROM b
LEFT JOIN b l1  ON l1.sensor_id  = b.sensor_id AND l1.hour_ist  = b.hour_ist - INTERVAL 1 HOURS
LEFT JOIN b l3  ON l3.sensor_id  = b.sensor_id AND l3.hour_ist  = b.hour_ist - INTERVAL 3 HOURS
LEFT JOIN b l6  ON l6.sensor_id  = b.sensor_id AND l6.hour_ist  = b.hour_ist - INTERVAL 6 HOURS
LEFT JOIN b l24 ON l24.sensor_id = b.sensor_id AND l24.hour_ist = b.hour_ist - INTERVAL 24 HOURS
WHERE b.pm25_avg IS NOT NULL AND b.temperature_c IS NOT NULL
  AND b.hour_ist >= TIMESTAMP '{LOG_FROM}'
  AND b.hour_ist < date_trunc('hour', current_timestamp() + INTERVAL 330 MINUTES)
""")

if spark.catalog.tableExists(LOG):
    live = live.join(spark.table(LOG).select("sensor_id", "hour_ist"), ["sensor_id", "hour_ist"], "left_anti")

pdf = live.toPandas()
print("new rows to forecast:", len(pdf))

if len(pdf) > 0:
    pdf["hour_ist"] = pd.to_datetime(pdf["hour_ist"])
    pdf["pred_pm25_6h"] = model.predict(pdf[FEATURES])
    pdf["target_hour_ist"] = pdf["hour_ist"] + pd.Timedelta(hours=6)
    pdf["logged_at"] = datetime.now(timezone.utc).replace(tzinfo=None)
    pdf["model_version"] = "frozen-2026-11-01-clean"
    out = pdf[["sensor_id", "location_id", "hour_ist", "target_hour_ist", "pm25_now", "pred_pm25_6h", "logged_at", "model_version"]]
    spark.createDataFrame(out).write.mode("append").saveAsTable(LOG)
    print("logged:", len(out))